In [1]:
%reload_ext autoreload
%autoreload 2
from alignment_v2.experiments import arglib
from alignment_v2.experiments.experiment import Experiment
from copy import copy 
import time
import torch
from tqdm import tqdm
import numpy as np
import scipy as sp
import sklearn
import torch.nn.functional as F
from torch import nn 
from torchvision.transforms import v2 as transforms

import matplotlib as mpl
from matplotlib import pyplot as plt

from alignment_v2.models.registry import get_model
from alignment_v2.datasets import get_dataset
from alignment_v2.experiments.registry import get_experiment
from alignment_v2 import utils, train, processing, plotting
from alignment_v2 import files

from alignment_v2 import processing, plotting
from alignment_v2.experiments.experiment import Experiment
from alignment_v2.train import progressive_dropout, progressive_dropout_train  # Import the custom dropout experiment function
#from alignment_v2.processing import progressive_dropout_experiment

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('using device: ', DEVICE)

using device:  cuda


train

In [ ]:
# Model and Dataset Setup
model_name = 'AlexNet' #'CNN2P2' #'AlexNet' #
dataset_name = 'ImageNet' #'MNIST'#
dropout_rate = 0
learning_rate = 0.01
weight_decay = 0
replicates = 1
optim = torch.optim.Adam

nets = [
    get_model(
        model_name,
        build=True,
        dataset=dataset_name,
        dropout=dropout_rate,
        ignore_flag=False,
    )
    for _ in range(replicates)
]
nets = [net.to(DEVICE) for net in nets]

optimizers = [optim(net.parameters(), lr=learning_rate, weight_decay=weight_decay) for net in nets]

loader_parameters = dict(
    shuffle=True,
)
dataset = get_dataset(dataset_name, build=True, transform_parameters=nets[0], loader_parameters=loader_parameters, device=DEVICE)

# Training Parameters
training_params = dict(
    num_epochs=20,
    alignment=True,
    alignment_expansion=False,
    compare_expected=False,
    frequency=1,
    delta_alignment=False,
)

# Step 1: Train the Model
print("Training the network...")
results = train.train(nets, optimizers, dataset, **training_params)

Training the network...


minibatch:  30%|██▉       | 186/626 [1:30:51<3:07:47, 25.61s/it]

prune and retrain

In [ ]:

from alignment_v2.train import progressive_dropout, progressive_dropout_train  # Import the custom dropout experiment function

def progressive_dropout_experiment(exp, nets, dataset, alignment=None, train_set=False, num_drops=10, by_layer=True, retrain=False):

    dropout_func = progressive_dropout_train if retrain else progressive_dropout
    
    # Targeted dropout experiment
    print("Performing targeted dropout" + (" with retraining..." if retrain else "..."))
    dropout_parameters = dict(num_drops=num_drops, by_layer=by_layer, train_set=train_set)
    dropout_results = dropout_func(nets, dataset, alignment=alignment, **dropout_parameters)    
    return dropout_results, dropout_parameters

# Set parameters for the dropout experiment
num_drops = 15
by_layer = False
train_set = False
retrain = False  # Set to True to perform retraining after pruning
exp = {'args': {'num_drops': num_drops, 'dropout_by_layer': by_layer}}

# Step 2: Perform Progressive Dropout Experiment
print("Performing progressive dropout based on alignment...")
dropout_params = {
    "num_drops": num_drops,      # Number of dropout levels to apply
    "by_layer": by_layer,        # Apply dropout by each layer
    "train_set": train_set,      # Use the test set for measuring dropout effects
    "retrain": retrain           # Enable retraining if required
}

if 1==1:

    # Run the dropout experiment and retrieve the results
    dropout_results, dropout_parameters = progressive_dropout_experiment(
        exp, nets, dataset, alignment=results["alignment"], **dropout_params
    )
    
    # # Run the dropout experiment and retrieve the results
    # dropout_results, dropout_parameters = progressive_dropout_experiment(
    #     exp, nets, dataset, alignment=None, **dropout_params
    # )

    # Separate results for before and after retraining if retraining was enabled
    if retrain:
        # Assuming progressive_dropout_train outputs pre- and post-training results as a tuple
        dropout_results_before = dropout_results["before"]
        dropout_results_after = dropout_results["after"]
    else:
        dropout_results_before = dropout_results
        dropout_results_after = None

    # Define experiment parameters for plotting
    prms = {
        "vals": [model_name],
        "name": "ModelType",
        "dataset": dataset_name,
        "dropout": dropout_rate,
        "lr": learning_rate,
        "weight_decay": weight_decay
    }

    print("Plotting dropout experiment results...")

    # Plotting both before and after retraining results
    plotting.plot_dropout_results(
        exp=None,  # replace with experiment name if needed
        dropout_results=dropout_results_before,
        dropout_parameters=dropout_parameters,
        prms=prms,
        dropout_type="alignment-based",
    )

    if retrain:
        plotting.plot_dropout_results(
            exp=None,  # replace with experiment name if needed
            dropout_results=dropout_results_after,
            dropout_parameters=dropout_parameters,
            prms=prms,
            dropout_type="alignment-based",
        )

if 1==2:
    ###############################

    # Run the dropout experiment and retrieve the results
    dropout_results, dropout_parameters = progressive_dropout_experiment(
        exp, nets, dataset, alignment=results["alignment_1"], **dropout_params
    )

    # Separate results for before and after retraining if retraining was enabled
    if retrain:
        # Assuming progressive_dropout_train outputs pre- and post-training results as a tuple
        dropout_results_before = dropout_results["before"]
        dropout_results_after = dropout_results["after"]
    else:
        dropout_results_before = dropout_results
        dropout_results_after = None

    # Define experiment parameters for plotting
    prms = {
        "vals": [model_name],
        "name": "ModelType",
        "dataset": dataset_name,
        "dropout": dropout_rate,
        "lr": learning_rate,
        "weight_decay": weight_decay
    }

    print("Plotting dropout experiment results...")

    # Plotting both before and after retraining results
    plotting.plot_dropout_results(
        exp=None,  # replace with experiment name if needed
        dropout_results=dropout_results_before,
        dropout_parameters=dropout_parameters,
        prms=prms,
        dropout_type="alignment-based",
    )

    if retrain:
        plotting.plot_dropout_results(
            exp=None,  # replace with experiment name if needed
            dropout_results=dropout_results_after,
            dropout_parameters=dropout_parameters,
            prms=prms,
            dropout_type="alignment-based",
        )

    # Run the dropout experiment and retrieve the results
    dropout_results, dropout_parameters = progressive_dropout_experiment(
        exp, nets, dataset, alignment=results["alignment_2"], **dropout_params
    )

    # Separate results for before and after retraining if retraining was enabled
    if retrain:
        # Assuming progressive_dropout_train outputs pre- and post-training results as a tuple
        dropout_results_before = dropout_results["before"]
        dropout_results_after = dropout_results["after"]
    else:
        dropout_results_before = dropout_results
        dropout_results_after = None

    # Define experiment parameters for plotting
    prms = {
        "vals": [model_name],
        "name": "ModelType",
        "dataset": dataset_name,
        "dropout": dropout_rate,
        "lr": learning_rate,
        "weight_decay": weight_decay
    }

    print("Plotting dropout experiment results...")

    # Plotting both before and after retraining results
    plotting.plot_dropout_results(
        exp=None,  # replace with experiment name if needed
        dropout_results=dropout_results_before,
        dropout_parameters=dropout_parameters,
        prms=prms,
        dropout_type="alignment-based",
    )

    if retrain:
        plotting.plot_dropout_results(
            exp=None,  # replace with experiment name if needed
            dropout_results=dropout_results_after,
            dropout_parameters=dropout_parameters,
            prms=prms,
            dropout_type="alignment-based",
        )

 
    # Run the dropout experiment and retrieve the results
    dropout_results, dropout_parameters = progressive_dropout_experiment(
        exp, nets, dataset, alignment=results["alignment_0"], **dropout_params
    )

    # Separate results for before and after retraining if retraining was enabled
    if retrain:
        # Assuming progressive_dropout_train outputs pre- and post-training results as a tuple
        dropout_results_before = dropout_results["before"]
        dropout_results_after = dropout_results["after"]
    else:
        dropout_results_before = dropout_results
        dropout_results_after = None

    # Define experiment parameters for plotting
    prms = {
        "vals": [model_name],
        "name": "ModelType",
        "dataset": dataset_name,
        "dropout": dropout_rate,
        "lr": learning_rate,
        "weight_decay": weight_decay
    }

    print("Plotting dropout experiment results...")

    # Plotting both before and after retraining results
    plotting.plot_dropout_results(
        exp=None,  # replace with experiment name if needed
        dropout_results=dropout_results_before,
        dropout_parameters=dropout_parameters,
        prms=prms,
        dropout_type="alignment-based",
    )

    if retrain:
        plotting.plot_dropout_results(
            exp=None,  # replace with experiment name if needed
            dropout_results=dropout_results_after,
            dropout_parameters=dropout_parameters,
            prms=prms,
            dropout_type="alignment-based",
        )

    ######################################################################
    al = results["alignment_0"] + results["alignment_red"]

    # Run the dropout experiment and retrieve the results
    dropout_results, dropout_parameters = progressive_dropout_experiment(
        exp, nets, dataset, alignment=al, **dropout_params
    )

    # Separate results for before and after retraining if retraining was enabled
    if retrain:
        # Assuming progressive_dropout_train outputs pre- and post-training results as a tuple
        dropout_results_before = dropout_results["before"]
        dropout_results_after = dropout_results["after"]
    else:
        dropout_results_before = dropout_results
        dropout_results_after = None

    # Define experiment parameters for plotting
    prms = {
        "vals": [model_name],
        "name": "ModelType",
        "dataset": dataset_name,
        "dropout": dropout_rate,
        "lr": learning_rate,
        "weight_decay": weight_decay
    }

    print("Plotting dropout experiment results...")

    # Plotting both before and after retraining results
    plotting.plot_dropout_results(
        exp=None,  # replace with experiment name if needed
        dropout_results=dropout_results_before,
        dropout_parameters=dropout_parameters,
        prms=prms,
        dropout_type="alignment-based",
    )

    if retrain:
        plotting.plot_dropout_results(
            exp=None,  # replace with experiment name if needed
            dropout_results=dropout_results_after,
            dropout_parameters=dropout_parameters,
            prms=prms,
            dropout_type="alignment-based",
        )


In [ ]:
alignment=results["alignment"]
print(len(alignment))

print(alignment[0].shape)
print(alignment[0][0,:,:])
#print(dropout_results.keys())
#print(dropout_results['progdrop_acc_rand'])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parameters
N = 10  # Number of dendrites
N_E = 100  # Excitatory synapses per dendrite
p_E = 0.1  # Excitatory activation probability
p_I = 0.1  # Inhibitory activation probability
r_d_values = [0, 0.5, 1, 2]  # Inhibitory to excitatory ratios
trials = 10000

# Simulation and Analysis
for r_d in r_d_values:
    N_I = int(r_d * N_E)
    y_s_values = []
    for _ in range(trials):
        y_d_values = []
        for _ in range(N):
            x_E = np.random.binomial(N_E, p_E)
            x_I = np.random.binomial(N_I, p_I)
            y_d = x_E / (1 + x_E + x_I)
            y_d_values.append(y_d)
        y_s = sum(y_d_values)
        y_s_values.append(y_s)
    # Plotting
    plt.hist(y_s_values, bins=50, alpha=0.6, label=f'r_d = {r_d}')
    plt.xlabel('Neuron Output (y_s)')
    plt.ylabel('Frequency')
    plt.title('Distribution of Neuron Output')
    plt.legend()
    plt.show()

In [ ]:
######################################################################
al = results["alignment"] + results["alignment_0"] + results["alignment_1"] + results["alignment_2"]

# Run the dropout experiment and retrieve the results
dropout_results, dropout_parameters = progressive_dropout_experiment(
    exp, nets, dataset, alignment=al, **dropout_params
)

# Separate results for before and after retraining if retraining was enabled
if retrain:
    # Assuming progressive_dropout_train outputs pre- and post-training results as a tuple
    dropout_results_before = dropout_results["before"]
    dropout_results_after = dropout_results["after"]
else:
    dropout_results_before = dropout_results
    dropout_results_after = None

# Define experiment parameters for plotting
prms = {
    "vals": [model_name],
    "name": "ModelType",
    "dataset": dataset_name,
    "dropout": dropout_rate,
    "lr": learning_rate,
    "weight_decay": weight_decay
}

print("Plotting dropout experiment results...")

# Plotting both before and after retraining results
plotting.plot_dropout_results(
    exp=None,  # replace with experiment name if needed
    dropout_results=dropout_results_before,
    dropout_parameters=dropout_parameters,
    prms=prms,
    dropout_type="alignment-based",
)

if retrain:
    plotting.plot_dropout_results(
        exp=None,  # replace with experiment name if needed
        dropout_results=dropout_results_after,
        dropout_parameters=dropout_parameters,
        prms=prms,
        dropout_type="alignment-based",
    )


In [ ]:

plt.scatter(results['alignment'][2],results['alignment_2'][2], s = 5)

In [ ]:
print(results.keys())
for i, tensor in enumerate(results['alignment']):
    print(f"Shape of tensor {i}: {tensor.shape}")
    

def plot_alignment_distribution(results):
    alignment = results['alignment']
    
    for i, tensor in enumerate(alignment):
        # Flatten the tensor for plotting to get all alignment values in one dimension
        dat = tensor.cpu().numpy()  # Convert to a 1D numpy array
        #print(dat.shape)
        data = np.mean(dat[0,:,:], axis=0)
        print(data.shape)
        #data = tensor.flatten().cpu().numpy()  # Convert to a 1D numpy array
        # Plot histogram for the distribution
        plt.figure(figsize=(10, 5))
        plt.hist(data, bins=30, color='blue', alpha=0.7)
        plt.title(f'Alignment Distribution for Layer {i + 1}')
        plt.xlabel('Alignment Value')
        plt.ylabel('Frequency')
        plt.show()

        # Plot boxplot for a quick look at distribution characteristics
        plt.figure(figsize=(10, 5))
        plt.boxplot(data, vert=False)
        plt.title(f'Alignment Boxplot for Layer {i + 1}')
        plt.xlabel('Alignment Value')
        plt.show()

# Call the function with results dictionary
plot_alignment_distribution(results)

In [ ]:
import matplotlib.pyplot as plt
import torch
import scipy.cluster.hierarchy as sch
import numpy as np

def plot_redundancy_matrix_for_each_net(results):
    alignment_red = results['alignment_red']

    # Iterate over each repetition (e.g., each network)
    for net_index, tensor in enumerate(alignment_red):
        #if net_index ==0:
        print(f"Plotting for Network {net_index + 1}")
        for la in range(tensor.shape[0]):
            if la==1:
                # Average over batches
                tensor_mean = torch.mean(tensor[la,:,:], dim=0)
                #print(tensor_mean.shape)
                # Verify if the tensor can be reshaped into a square matrix
                total_elements = tensor_mean.shape[0]
                n = int(total_elements ** 0.5)  # Calculate size n based on total elements

                if n * n != total_elements:
                    print(f"Skipping Network {net_index + 1}: cannot reshape tensor of size {total_elements} into a square matrix.")
                    continue

                # Reshape the averaged redundancy matrix to (n, n)
                redundancy_matrix = tensor_mean[:n*n].reshape(n, n).cpu().numpy()

                # # Plot the redundancy matrix as a heatmap
                # plt.figure(figsize=(8, 6))
                # plt.imshow(redundancy_matrix, cmap='viridis', aspect='equal')
                # plt.colorbar(label='Redundancy Value')
                # plt.title(f'Averaged Redundancy Matrix for layer {net_index + 1}')
                # plt.xlabel('Node Index')
                # plt.ylabel('Node Index')
                # plt.show()
                
                                # Perform hierarchical clustering on the redundancy matrix
                linkage_matrix = sch.linkage(redundancy_matrix, method='single')
                dendro = sch.dendrogram(linkage_matrix, no_plot=True)
                reordered_indices = dendro['leaves']
                clustered_matrix = redundancy_matrix[reordered_indices, :][:, reordered_indices]

                # Plot the clustered redundancy matrix as a heatmap
                plt.figure(figsize=(10, 8))
                plt.imshow(clustered_matrix, cmap='viridis', aspect='equal')
                plt.colorbar(label='Redundancy Value')
                plt.title(f'Clustered Redundancy Matrix for Network {net_index + 1}, Layer {la + 1}')
                plt.xlabel('Node Index (Clustered)')
                plt.ylabel('Node Index (Clustered)')
                plt.show()

# Call the function with the results dictionary
plot_redundancy_matrix_for_each_net(results)

In [ ]:
images = []
labels = []
for batch in tqdm(dataset.train_loader):
    cimages, clabels = dataset.unwrap_batch(batch)
    images.append(cimages)
    labels.append(clabels)
images = torch.concatenate(images, dim=0)
images = images.view(images.size(0), -1)
# images = images - images.mean(dim=0)
labels = torch.concatenate(labels, dim=0)
print(images.shape, labels.shape)

# get stacked indices to the elements of each class
classes = torch.unique(labels)
num_classes = len(classes)
idx_to_class = [torch.where(labels == ii)[0] for ii in classes]
num_per_class = [len(idx) for idx in idx_to_class]
min_per_class = min(num_per_class)
if any([npc > min_per_class for npc in num_per_class]):
    idx_to_class = [idx[:min_per_class] for idx in idx_to_class]

# use single tensor for fast indexing
idx_to_class = torch.stack(idx_to_class).unsqueeze(2)

# (classes, images, image_dimension)
images_by_class = torch.gather(images.unsqueeze(0).expand(num_classes, -1, -1), 1, idx_to_class.expand(-1, -1, images.size(1)))

In [ ]:
# Make a CV Dataset of inputs to each layer for each class
inputs_to_layers = net.get_layer_inputs(images, precomputed=False)
inputs_to_layers = net._preprocess_inputs(inputs_to_layers)
inputs_by_class = [torch.gather(inputs.unsqueeze(0).expand(num_classes, -1, -1), 1, idx_to_class.expand(-1, -1, inputs.size(1))) 
                   for inputs in inputs_to_layers]

print([i.shape for i in inputs_to_layers])
print([i.shape for i in inputs_by_class])
print(images_by_class.shape)

In [ ]:
# make a CV dataset
num_per_class = images_by_class.size(1)
random_sort = torch.randperm(num_per_class)
train_examples = random_sort[:num_per_class//2]
test_examples = random_sort[num_per_class//2:num_per_class//2+num_per_class//2]
inputs_train = [inputs[:, train_examples].reshape(-1, inputs.size(2)) for inputs in inputs_by_class]
inputs_test = [inputs[:, test_examples].reshape(-1, inputs.size(2)) for inputs in inputs_by_class]

# measure eigenfeatures
w, v = utils.named_transpose([utils.smart_pca(inputs.T) for inputs in inputs_to_layers])
w = [(w / torch.sum(w)).cpu() for w in w]

# cvPCA method
cc = [utils.shuff_cvPCA(itrain.T.cpu(), itest.T.cpu(), nshuff=5).cpu() for itrain, itest in zip(inputs_train, inputs_test)]
cc = [cc.mean(dim=0) for cc in cc]
cc = [cc / torch.sum(cc) for cc in cc]

In [ ]:
cumsum = True

num_layers = len(w)
figdim = 3
fig, ax = plt.subplots(1, num_layers, figsize=(num_layers*figdim, figdim), layout='constrained')
for layer in range(num_layers):
    if cumsum:
        ax[layer].plot(range(len(w[layer])), torch.cumsum(w[layer], 0), c='k', label='all data')
        ax[layer].plot(range(len(w[layer])), torch.cumsum(cc[layer], 0), c='r', label='cvPCA')
    else:
        ax[layer].plot(range(len(w[layer])), w[layer], c='k', label='all data')
        ax[layer].plot(range(len(w[layer])), cc[layer], c='r', label='cvPCA')
        ax[layer].set_yscale('log')
        ax[layer].set_ylim(1e-4)

    ax[layer].set_xscale('log')
    ax[layer].set_title(f"Layer {layer}")
    ax[layer].legend()
    
plt.show()

# I'd like to compare these cvPCA results to the observed alignment distribution
# to determine if the reason alignment and delta alignment is much higher than expected
# according to Ila Fiete is because of the batching implicitly "cross-validating" the 
# updates according to class-dependent structure rather than simply all structure. 

# In other words, if it is the case that the network primarily learns stimulus specific
# dimensions, rather than full data dimensions, it may be because any other dimensions
# are ignored by the batching. (And therefore implicitly prevents over-generalization!)

# Other experiment: Determine if these dimensions are overrepresented in the dataset. 

In [ ]:
# Measure distribution of alignment, compare with expected given "Alignment" from Fiete definition
calign_bins = torch.linspace(0, 0.2, 301)
calign_centers = utils.edge2center(calign_bins)
c_alignment = net.measure_alignment(images, precomputed=False, method="alignment")
c_dist = [utils.expected_alignment_distribution(ev, valid_rotation=False, bins=calign_bins)[0] for ev in w]
t_dist = [torch.histogram(align.cpu(), bins=calign_bins, density=True)[0] for align in c_alignment]
cv_dist = [utils.expected_alignment_distribution(ev, valid_rotation=False, bins=calign_bins)[0] for ev in cc]


In [ ]:
# max_bin = [torch.where(torch.any(cc>0, dim=0))[0][-1] for cc in cad]

num_layers = len(w)
figdim = 3
fig, ax = plt.subplots(1, num_layers, figsize=(num_layers*figdim, figdim), layout='constrained')
for layer in range(num_layers):
    ax[layer].plot(calign_centers, c_dist[layer], c='k', label='expected distribution')
    ax[layer].plot(calign_centers, cv_dist[layer], c='b', label='expected cv-distribution')
    ax[layer].plot(calign_centers, t_dist[layer], c='r', label='true distribution')
    # ax[layer].set_xscale('log')
    ax[layer].set_title(f"Layer {layer}")
    ax[layer].legend()
    
plt.show()

# I'd like to compare these cvPCA results to the observed alignment distribution
# to determine if the reason alignment and delta alignment is much higher than expected
# according to Ila Fiete is because of the batching implicitly "cross-validating" the 
# updates according to class-dependent structure rather than simply all structure. 

# In other words, if it is the case that the network primarily learns stimulus specific
# dimensions, rather than full data dimensions, it may be because any other dimensions
# are ignored by the batching. (And therefore implicitly prevents over-generalization!)

# Other experiment: Determine if these dimensions are overrepresented in the dataset. 

In [ ]:
[torch.max(align) for align in c_alignment]

In [ ]:
# Some Code for continual learning with permuted MNIST
from functools import partial

def permute(batch, batch_dim=True, shuffle_idx=None):
    if shuffle_idx is not None:
        original_size = batch[0].shape
        if batch_dim:
            batch[0] = batch[0][:, shuffle_idx]
        else:
            batch[0] = batch[0][shuffle_idx]
        batch[0].reshape(original_size)
    return batch

def add_permutation(dataset, num_pixels=784):
    perm = partial(permute, batch_dim=True, shuffle_idx=torch.randperm(num_pixels))
    if dataset.extra_transform is None:
        dataset.extra_transform = []
    dataset.extra_transform.append(perm)
    return dataset

def update_permutation(dataset, num_pixels=784):
    perm = partial(permute, batch_dim=True, shuffle_idx=torch.randperm(num_pixels))
    dataset.extra_transform[-1] = perm
    return dataset

def do_permuted_round(nets, optimizers, dataset, train_epochs=1, verbose=False):
    parameters = dict(
        verbose=verbose,
        num_epochs=train_epochs,
        alignment=False,
    )
    dataset = update_permutation(dataset)
    
    # train and test
    train_results = train.train(nets, optimizers, dataset, **parameters)
    test_results = train.test(nets, dataset, **parameters)

    return train_results, test_results

In [ ]:
model_name = 'MLP'
dataset_name = 'MNIST'

hidden_widths = [2000, 2000, 2000]

lrs = [3e-3, 1e-3, 3e-4, 1e-4]
num_replicates = 3

nets = []
optimizers = []
net_lr = []
for lr in lrs:    
    for rep in range(num_replicates):
        net = get_model(model_name, build=True, dataset=dataset_name, hidden_widths=hidden_widths, dropout=0.0, ignore_flag=False)
        net.to(DEVICE)

        optimizer = torch.optim.Adam(net.parameters(), lr=lr)

        nets.append(net)
        optimizers.append(optimizer)
        net_lr.append(lr)

loader_parameters = dict(
    shuffle=True,
    batch_size=10,
)
dataset = get_dataset(dataset_name, build=True, transform_parameters=net, loader_parameters=loader_parameters, device=DEVICE)
dataset = add_permutation(dataset)

num_rounds = 10

train_loss = []
train_accuracy = []
test_loss = []
test_accuracy = []
for round in tqdm(range(num_rounds)):
    c_train_res, c_test_res = do_permuted_round(nets, optimizers, dataset)
    train_loss.append(c_train_res['loss'])
    train_accuracy.append(c_train_res['accuracy'])
    test_loss.append(c_test_res['loss'])
    test_accuracy.append(c_test_res['accuracy'])

In [ ]:
loss = torch.stack([torch.tensor(l) for l in test_loss])
accuracy = torch.stack([torch.tensor(a) for a in test_accuracy])

type_loss = utils.compute_stats_by_type(loss, len(lrs), 1)[0]
type_accuracy = utils.compute_stats_by_type(accuracy, len(lrs), 1)[0]

# print(loss.shape, accuracy.shape, type_loss.shape, type_accuracy.shape)

cols = mpl.colormaps['Set1'].resampled(len(lrs))
names = [f"lr={lr}" for lr in lrs]

fig, ax = plt.subplots(1, 2, figsize=(6, 3), layout='constrained')
for ii in range(1, len(lrs)):
    ax[0].plot(range(num_rounds), type_loss[:, ii], c=cols(ii), label=names[ii])
    ax[1].plot(range(num_rounds), type_accuracy[:, ii], c=cols(ii), label=names[ii])

ax[0].set_xlabel('Rounds of Permuted MNIST')
ax[1].set_xlabel('Rounds of Permuted MNIST')
ax[0].set_ylabel('loss')
ax[1].set_ylabel('accuracy')
ax[0].legend(loc='best')
ax[1].legend(loc='best')
plt.show()